# 05 Data integration

Build one feature row per recorded transaction from the cleaned Parquet tables. The [prediction design](../reports/prediction_design.md) defines the event clock and key `(batch_id, leg, event_type)`. Use the existing split manifest to verify identity, time and split coverage. The target remains in its separate Parquet file.

## What may enter the feature table

| Source | Fields and use |
| --- | --- |
| Transactions | The seven [approved current-event inputs](../reports/feature_audit.csv) are eligible under the design's timing assumption. `batch_id` and `timestamp` are metadata for joining and ordering. |
| Batches | Use `batch_id`, `medicine_id`, and `manufacturer_id` for joins. Use `production_date` to reject events dated before their batches. Keep these batch details outside model inputs. The release gives no recorded availability history for fields such as quantity. |
| Medicines | Admit `storage_min`, `storage_max`, and `is_cold_chain` as fixed synthetic catalogue specifications after checking the release. The code creates these before events and uses them in event generation. Verify the fixed values against configuration and reproduce the stored excursion rate from current and earlier readings. Temperature, the stored rate, and event sequence are used only for this verification. |
| Entities | Use `entity_id` and `entity_type` to confirm that the referenced entity is a manufacturer. |
| Fraud outcomes | Read only the allowed transaction projections. The fraud table and all outcome columns stay outside the integration. |

The generator supports availability of the fixed storage rules **in this offline simulation**. The audit does not establish when a live feed would expose the catalogue or the event readings. A changing field would need versioned records with both effective dates and the times those versions became known. A point-in-time join would require an effective and already available version at each transaction timestamp. TRACE-X has no usable version history for licence, compliance, audit, certificate or price snapshots, so these fields stay out.

[Datasheet, storage rules](../Phase1_Dataset/DATASHEET.md#L75) · [Generator, catalogue](../Phase1_Dataset/code/generator.py#L123) · [Generator, event use](../Phase1_Dataset/code/generator.py#L225) · [Generator, expiry calculation inputs and excursion history](../Phase1_Dataset/code/generator.py#L320)

In [ ]:
from hashlib import file_digest
from pathlib import Path
import json
import sys

import pandas as pd
import pyarrow.parquet as pq
import yaml
from IPython.display import display

current = Path.cwd()
root = next(path for path in (current, *current.parents) if (path / "AGENTS.md").is_file())
sys.path.insert(0, str(root))
sys.dont_write_bytecode = True

from src.integration import (
    BATCH_READ_COLUMNS, ENTITY_READ_COLUMNS, EVENT_FEATURES, FEATURE_COLUMNS, MEDICINE_FEATURES,
    MEDICINE_READ_COLUMNS, OUTPUT_COLUMNS, PREDICTION_TIMESTAMP, TRANSACTION_KEY,
    TRANSACTION_READ_COLUMNS, IntegrationError, checked_lookup, integrate_transactions,
    require_unique_key, verify_event_timing,
)

processed = root / "processed"
cleaned = processed / "cleaned"
reports = root / "reports"
boundary = json.loads((processed / "split_boundaries.json").read_text(encoding="utf-8"))
manifest_path = processed / boundary["assignment_file"]
with manifest_path.open("rb") as manifest_file:
    assert file_digest(manifest_file, "sha256").hexdigest() == boundary["assignment_sha256"]
config = yaml.safe_load((root / "Phase1_Dataset" / "code" / "config.yaml").read_text(encoding="utf-8"))
audit = pd.read_csv(reports / "feature_audit.csv")

assert list(TRANSACTION_KEY) == boundary["transaction_key"]
assert PREDICTION_TIMESTAMP == boundary["prediction_timestamp"]
safe = audit.loc[audit["category"].eq("Safe input features")]
assert set(EVENT_FEATURES) == set(safe.loc[
    safe["table"].eq("transactions") & safe["planned_use"].isin(["current_event_input", "current_event_input_and_key"]),
    "column",
])
assert set(MEDICINE_FEATURES).issubset(safe.loc[
    safe["table"].eq("medicines") & safe["planned_use"].eq("conditional_lookup"), "column"
])
print("Verified the fixed split manifest and the integration allowlist.")

## Check that invalid joins fail

These small examples exercise the same reusable functions. A repeated parent key can multiply events, an unmatched key can leave an event without context, and a batch dated later than its event can leak future information. Each case must stop the integration.

In [ ]:
def expect_failure(action, fragment):
    try:
        action()
    except IntegrationError as error:
        assert fragment in str(error), str(error)
    else:
        raise AssertionError(f"Expected a failure containing {fragment!r}")


left = pd.DataFrame({"batch_id": ["B1", "B1"]})
parent = pd.DataFrame({"batch_id": ["B1"], "medicine_id": ["M1"]})
joined, test_audit = checked_lookup(left, parent, left_key="batch_id", relationship="example")
assert joined["medicine_id"].tolist() == ["M1", "M1"]
assert test_audit["rows_before"] == test_audit["rows_after"] == 2

expect_failure(
    lambda: checked_lookup(left, pd.concat([parent, parent]), left_key="batch_id", relationship="example"),
    "duplicate keys",
)
expect_failure(
    lambda: checked_lookup(left, parent.iloc[:0], left_key="batch_id", relationship="example"),
    "unmatched rows",
)
expect_failure(
    lambda: checked_lookup(pd.DataFrame({"batch_id": [None]}), parent, left_key="batch_id", relationship="example"),
    "null foreign keys",
)
expect_failure(
    lambda: verify_event_timing(pd.DataFrame({
        "timestamp": pd.to_datetime(["2025-05-01T12:00:00"]),
        "production_date": pd.to_datetime(["2025-05-02"]),
    })),
    "before batch production",
)
expect_failure(
    lambda: require_unique_key(pd.DataFrame({"batch_id": ["B1", "B1"], "leg": [0, 0],
                                             "event_type": ["shipment", "shipment"]}),
                               TRANSACTION_KEY, "example transactions"),
    "duplicate keys",
)
print("Verified the join, duplicate-key, unmatched-key and future-date assertions.")

## Run and verify the integration

Join entity and medicine records to batches first, then attach the validated batch lookup to transactions. The three links must be many-to-one. An extra or unmatched row stops the run. Compare the final transaction identities and clocks against the fixed manifest before publishing the feature table. The manifest still controls which rows are used for training, validation and test.

In [ ]:
def read_cleaned(name, columns):
    return pd.read_parquet(cleaned / f"{name}.parquet", columns=list(columns))


events = read_cleaned("transactions", TRANSACTION_READ_COLUMNS)
batches = read_cleaned("batches", BATCH_READ_COLUMNS)
medicines = read_cleaned("medicines", MEDICINE_READ_COLUMNS)
entities = read_cleaned("entities", ENTITY_READ_COLUMNS)

integrated, joins, timing = integrate_transactions(events, batches, medicines, entities, config["storage_ranges"])
assert len(events) == len(integrated) == boundary["source_rows"]
assert timing["preproduction_transactions"] == 0 and timing["storage_rate_matches"] == len(integrated)
assert integrated.columns.tolist() == list(OUTPUT_COLUMNS)
assert set(FEATURE_COLUMNS).isdisjoint({boundary["target_column"], "batch_id", PREDICTION_TIMESTAMP})
forbidden_roles = ["Target and label-related columns", "Future-information columns", "Suspected synthetic leakage columns"]
forbidden = set(audit.loc[audit["category"].isin(forbidden_roles), "column"]) - {PREDICTION_TIMESTAMP}
assert set(integrated.columns).isdisjoint(forbidden)
assert not integrated[list(FEATURE_COLUMNS)].isna().any().any()

identity = [*TRANSACTION_KEY, PREDICTION_TIMESTAMP]
fixed = pd.read_parquet(manifest_path, columns=[*identity, "split"])
require_unique_key(fixed, TRANSACTION_KEY, "split manifest")
pd.testing.assert_frame_equal(
    integrated[identity].sort_values(identity, ignore_index=True),
    fixed[identity].sort_values(identity, ignore_index=True),
    check_dtype=False,
)
split_counts = fixed["split"].value_counts().to_dict()
assert split_counts == {name: summary["rows"] for name, summary in boundary["split_summary"].items()}
join_report = pd.DataFrame(joins)
display(join_report)
display(pd.Series(timing, name="rows or catalogue records").to_frame())
display(pd.Series(split_counts, name="verified rows").rename_axis("frozen assignment").to_frame())
print(f"Verified {len(integrated):,} unique transactions against the frozen splits.")

## Record the results

The integrated DataFrame is named `integrated` and the reusable function returns it. The report lists the exact feature names and keys for later notebooks. The validated frame contains every assignment, including boundary-spanning rows. For modelling, select the required split through the frozen manifest.

In [ ]:
report_path = reports / "data_integration.md"
join_lines = [
    "| Join | Relationship | Rows before | Rows after | Unmatched keys |",
    "| --- | --- | ---: | ---: | ---: |",
]
join_lines.extend(
    f"| {row['join']} | {row['relationship']} | {row['rows_before']:,} | {row['rows_after']:,} | {row['unmatched_foreign_keys']:,} |"
    for row in joins
)
feature_names = ", ".join(f"`{name}`" for name in FEATURE_COLUMNS)
text = [
    "# Transaction data integration",
    f"The [executed notebook](../notebooks/05_Data_Integration.ipynb) built an integrated DataFrame "
    f"with {len(integrated):,} rows and {len(OUTPUT_COLUMNS)} columns. There is one row per `(batch_id, leg, event_type)`. "
    "The original transaction identities and timestamps match the frozen split manifest exactly.",
    "## Join validation",
    "\n".join(join_lines),
    "Each parent key was complete and unique before the join. No many-to-many expansion occurred, "
    "and there were zero duplicate transaction keys before or after the joins. "
    "The entity lookup confirmed manufacturer roles.",
    "## Feature fields and timing",
    f"The model input list is {feature_names}. Keep `batch_id` and `timestamp` as metadata. "
    "Batch and entity lookups establish relationships and validate the event context. Their extra fields stay outside the feature table.",
    f"All {timing['catalogue_records_checked']:,} medicine records matched their configured storage rules. The generator creates the "
    "catalogue before events. In the released data, those limits reproduce the recorded storage excursion rate for "
    f"all {timing['storage_rate_matches']:,} rows using only current and earlier batch events. "
    f"There were {timing['preproduction_transactions']} events dated before batch production. This supports using the fixed storage "
    "specifications in the offline simulation. It does not verify live data delivery. [Generator](../Phase1_Dataset/code/generator.py#L320)",
    "The release has no effective-date and availability history for changing fields. Licence, audit, compliance, certificates and price "
    "were withheld. A future versioned join must enforce that the version was both effective and known at scoring time. "
    "Production date, historical excursion calculations, and the other audit-only fields also stayed outside the feature table.",
    "## Split and label handling",
    f"The matched manifest records {split_counts['train']:,} training, {split_counts['validation']:,} validation, "
    f"{split_counts['test']:,} test, and {split_counts['excluded_boundary_batch']:,} boundary-excluded rows. "
    "Use the frozen assignments to select each split. All fraud outcomes remain separate, including the binary target. "
    "There was no feature-target analysis or model fitting.",
    "Failure-path tests verified that duplicate parents, null or unmatched foreign keys, duplicate transaction IDs, and future batch dates "
    "stop the run. Disk space prevented a standalone Parquet export. The notebook creates the full DataFrame in memory.",
]
report_path.write_text("\n\n".join(text) + "\n", encoding="utf-8")
print(f"Saved {report_path.relative_to(root)}. All integration assertions passed.")